# 🎮 Recommender System Demo using Matrix Factorization (NMF)

This notebook demonstrates a simple **video game recommender system** built with the **Surprise** library. It uses matrix factorization (NMF) to predict user–game ratings and recommend top games.

In [ ]:
# =======================================
# STEP 1: Import libs & read csv file
# =======================================

# Uncomment if running for the first time:
# !pip install scikit-surprise

import pandas as pd
import numpy as np
from collections import defaultdict
from surprise import NMF, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split

# Load dataset
df_games = pd.read_csv('./best_selling_video_games.csv')

# Assign a clean integer game ID
df_games['gameId'] = range(1, len(df_games) + 1)

# Preview dataset
df_games.head()

In [ ]:
# =======================================
# STEP 2: Simulate Synthetic User Ratings
# =======================================

# Since the dataset has no user ratings, we generate them:
#   - 500 synthetic users
#   - Each user rates 15–35 random games
#   - Base rating is scaled from sales figures (1–5 scale)
#   - Personal noise of ±1.0 added per rating

np.random.seed(42)

N_USERS   = 500  # number of synthetic users
MIN_RATED = 15   # minimum games rated per user
MAX_RATED = 35   # maximum games rated per user

# Normalise sales to a 1–5 base rating
min_sales = df_games['Sales'].min()
max_sales = df_games['Sales'].max()
df_games['BaseRating'] = 1 + 4 * (df_games['Sales'] - min_sales) / (max_sales - min_sales)

records = []
for user_id in range(1, N_USERS + 1):
    n_to_rate     = np.random.randint(MIN_RATED, MAX_RATED + 1)
    sampled_games = df_games.sample(n=n_to_rate, replace=False)
    for _, row in sampled_games.iterrows():
        noise  = np.random.uniform(-1.0, 1.0)
        rating = float(np.clip(round(row['BaseRating'] + noise, 1), 1.0, 5.0))
        records.append({'userId': user_id, 'gameId': row['gameId'], 'rating': rating})

ratings_df = pd.DataFrame(records)

# Preview ratings
ratings_df.head()

In [ ]:
# =======================================
# STEP 3: Prepare data for Surprise
# =======================================

# Prepare data for Surprise
reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(ratings_df[['userId', 'gameId', 'rating']], reader)

# Split data into training and testing sets
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# =============================================
# STEP 4: Train matrix factorization model (NMF)
# ==============================================

# NMF key parameters:
#   n_factors : number of latent factors (default 15)
#   n_epochs  : number of training iterations (default 50)
#   biased    : add user/item bias terms (default False)

model = NMF(n_factors=15, n_epochs=50, biased=False, random_state=42)
model.fit(trainset)

In [ ]:
# ===================================
# STEP 5: Evaluate on test set
# ===================================

predictions = model.test(testset)
rmse = accuracy.rmse(predictions)

# Convert test predictions into dictionary format for evaluation
user_actual = defaultdict(list)
user_pred   = defaultdict(list)

# Build lists of actual and predicted ratings per user
for uid, iid, true_r, est, _ in predictions:
    user_actual[uid].append((iid, true_r))
    user_pred[uid].append((iid, est))

# Parameters
K         = 5    # top-K cutoff
threshold = 4.0  # rating threshold to consider "liked"

# Initialize metrics
precision_list, recall_list = [], []

# Compute metrics per user
for uid in user_actual.keys():
    actual_items = [iid for iid, rating in user_actual[uid] if rating >= threshold]
    pred_sorted  = sorted(user_pred[uid], key=lambda x: x[1], reverse=True)
    pred_items   = [iid for iid, _ in pred_sorted]

    # Top-K predictions
    top_k_pred     = pred_items[:K]
    true_positives = len(set(top_k_pred) & set(actual_items))
    precision      = true_positives / K if K else 0
    recall         = true_positives / len(actual_items) if actual_items else 0

    # Append metrics
    precision_list.append(precision)
    recall_list.append(recall)

# Compute averages
results = {
    "Precision@5": np.mean(precision_list),
    "Recall@5":    np.mean(recall_list)
}

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# =====================================================
# STEP 6: Generate top-N recommendations for a few users
# ======================================================

def get_top_n(predictions, n=5):
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

top_n = get_top_n(predictions, n=5)

# Display sample recommendations
for uid, user_ratings in list(top_n.items())[:3]:
    print(f'\nTop recommendations for user {uid}:')
    for (iid, rating) in user_ratings:
        title = df_games.loc[df_games['gameId'] == iid, 'Title'].values
        title = title[0] if len(title) > 0 else 'Unknown'
        print(f'  {title} (predicted rating: {rating:.2f})')

In [ ]:
# ==============================================================
# STEP 7: Predict ratings for unseen games for a specific user
#
# What This Code Does:
#   - Finds which games a user hasn't rated.
#   - Uses the NMF model's predict() method to estimate ratings for each unseen game.
#   - Sorts them by predicted rating.
#   - Prints the top 5 highest predicted games.
# ==============================================================

def recommend_for_user(user_id, ratings_df, df_games, model, n=5):
    """
    Recommend top-N games for a given user based on predicted ratings.
    """
    # All games
    all_game_ids = df_games['gameId'].unique()

    # Games already rated by the user
    rated_game_ids = ratings_df[ratings_df['userId'] == user_id]['gameId'].unique()

    # Games not yet rated
    unseen_game_ids = [gid for gid in all_game_ids if gid not in rated_game_ids]

    # Predict ratings for unseen games
    preds = []
    for gid in unseen_game_ids:
        pred = model.predict(user_id, gid)
        preds.append((gid, pred.est))

    # Sort by estimated rating, descending
    preds.sort(key=lambda x: x[1], reverse=True)

    # Get top-N
    top_n = preds[:n]

    # Display results
    print(f"\n🎮 Top {n} recommended games for user {user_id}:")
    for gid, rating in top_n:
        title = df_games.loc[df_games['gameId'] == gid, 'Title'].values
        title = title[0] if len(title) > 0 else "Unknown"
        print(f"  {title} (predicted rating: {rating:.2f})")


# Example: Get recommendations for user IDs 1, 50, and 100
for uid in [1, 50, 100]:
    recommend_for_user(user_id=uid, ratings_df=ratings_df, df_games=df_games, model=model, n=5)